In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")

def load_bars(tf):
    b = pd.read_sql(
        "SELECT ts_utc, open, high, low, close, atr_14, session FROM fx_bars "
        "WHERE pair = 'EURUSD' AND timeframe = %(tf)s ORDER BY ts_utc",
        engine, params={"tf": tf}, index_col="ts_utc",
    )
    b.index = pd.to_datetime(b.index, utc=True)
    return b

b15 = load_bars("15m")
b60 = load_bars("1h")
b15 = b15[b15.index < "2022-01-01"]     # BUILD PERIOD ONLY
b60 = b60[b60.index < "2022-01-01"]
print(len(b15), len(b60))

174071 43527


In [4]:
import numpy as np
import pandas as pd

PIP = 0.0001
COST_PIPS = 1.4          # spread + slippage, round trip (from the specs)

def simulate_trade(bars, entry_i, side, stop, target, max_bars=None, exit_time=None):
    """
    bars:      DataFrame with open/high/low/close, index = bar start time (UTC)
    entry_i:   integer position of the bar we ENTER on (fill at its open)
    side:      +1 long, -1 short
    stop, target: absolute prices
    max_bars:  close at that bar's close if still open after this many bars
    exit_time: 'HH:MM' UTC, close at the open of the first bar at/after this time
    Returns a dict describing the trade, with P&L in pips after costs.
    """
    entry_px = bars["open"].iat[entry_i]
    n = len(bars)
    for k, i in enumerate(range(entry_i, n)):
        t = bars.index[i]
        o, h, l, c = bars["open"].iat[i], bars["high"].iat[i], bars["low"].iat[i], bars["close"].iat[i]

        # time exit: leave at this bar's open
        if exit_time is not None and k > 0 and t.strftime("%H:%M") >= exit_time:
            exit_px, reason = o, "time"
            break

        if side == 1:
            hit_stop, hit_tgt = l <= stop, h >= target
        else:
            hit_stop, hit_tgt = h >= stop, l <= target

        if hit_stop:                       # checked FIRST: worst case if both touched
            # if the bar opened beyond the stop (a gap), we fill at the open, not the stop
            gap = (o < stop) if side == 1 else (o > stop)
            exit_px, reason = (o if gap else stop), "stop"
            break
        if hit_tgt:
            exit_px, reason = target, "target"
            break
        if max_bars is not None and k + 1 >= max_bars:
            exit_px, reason = c, "max_bars"
            break
    else:
        exit_px, reason, i = bars["close"].iat[n - 1], "end_of_data", n - 1

    gross = side * (exit_px - entry_px) / PIP
    return {
        "entry_time": bars.index[entry_i], "exit_time": bars.index[i],
        "side": side, "entry": entry_px, "exit": exit_px, "reason": reason,
        "bars_held": i - entry_i + 1,
        "pips_gross": gross, "pips_net": gross - COST_PIPS,
        "risk_pips": abs(entry_px - stop) / PIP,
    }

In [5]:
b60 = b60.copy()
b60["ma20"] = b60["close"].rolling(20).mean()
b60["sd20"] = b60["close"].rolling(20).std()
# ATR rank vs the PREVIOUS 60 days of hourly bars (shift(1) means past only)
b60["atr_cap"] = b60["atr_14"].shift(1).rolling(60 * 24, min_periods=500).quantile(0.80)
b60[["close", "ma20", "sd20", "atr_14", "atr_cap"]].dropna().tail()

,close,ma20,sd20,atr_14,atr_cap
ts_utc,,,,,
2021-12-31 17:00:00+00:00,1.13862,1.133099,0.002299,0.001786,0.001457
2021-12-31 18:00:00+00:00,1.13846,1.133404,0.002583,0.001844,0.001460
2021-12-31 19:00:00+00:00,1.13825,1.133705,0.002782,0.001772,0.001461
2021-12-31 20:00:00+00:00,1.13760,1.133951,0.002901,0.001784,0.001461
2021-12-31 21:00:00+00:00,1.13686,1.134196,0.002932,0.001794,0.001462


In [6]:
def run_strategy_b(b60, band=2.0, stop_mult=1.5, max_bars=8):
    trades, skipped = [], {"no_data": 0, "high_vol": 0, "target_wrong_side": 0}
    upper = b60["ma20"] + band * b60["sd20"]
    lower = b60["ma20"] - band * b60["sd20"]
    i, n = 0, len(b60)
    while i < n - 1:
        t = b60.index[i]
        c = b60["close"].iat[i]
        side = 1 if c < lower.iat[i] else (-1 if c > upper.iat[i] else 0)
        if side == 0 or t.hour >= 7:                  # signal bar must START in 00:00-06:00 UTC
            i += 1; continue
        atr, cap, ma = b60["atr_14"].iat[i], b60["atr_cap"].iat[i], b60["ma20"].iat[i]
        if np.isnan(atr) or np.isnan(cap) or np.isnan(ma):
            skipped["no_data"] += 1; i += 1; continue
        if atr > cap:
            skipped["high_vol"] += 1; i += 1; continue
        entry_i = i + 1                                # fill at the NEXT bar's open
        entry_px = b60["open"].iat[entry_i]
        target = ma                                    # fixed at signal time
        if side * (target - entry_px) <= 0:            # already snapped back by the open
            skipped["target_wrong_side"] += 1; i += 1; continue
        stop = entry_px - side * stop_mult * atr
        tr = simulate_trade(b60, entry_i, side, stop, target, max_bars=max_bars)
        trades.append(tr)
        i = b60.index.get_loc(tr["exit_time"]) + 1     # one position at a time
    return pd.DataFrame(trades), skipped

tradesB, skippedB = run_strategy_b(b60)
print(skippedB)
t = tradesB.copy()
t["R"] = t["pips_net"] / t["risk_pips"]
t["R_gross"] = t["pips_gross"] / t["risk_pips"]
print(len(t), "trades | win rate", f"{(t.pips_net > 0).mean():.1%}",
      "| avg R", round(t.R.mean(), 3), "| avg R before costs", round(t.R_gross.mean(), 3))
print(t["reason"].value_counts(normalize=True).round(3))
print(t.groupby(t.entry_time.dt.year)[["R", "R_gross"]].mean().round(3))

{'no_data': 0, 'high_vol': 64, 'target_wrong_side': 0}
189 trades | win rate 42.9% | avg R -0.133 | avg R before costs -0.043
reason
stop        0.492
target      0.333
max_bars    0.175
Name: proportion, dtype: float64
                R  R_gross
entry_time                
2015       -0.177   -0.114
2016       -0.258   -0.177
2017       -0.081    0.003
2018        0.185    0.261
2019       -0.088    0.049
2020       -0.436   -0.351
2021       -0.079    0.029


In [7]:
grid = []
for band in [1.5, 2.0, 2.5]:
    for s in [1.0, 1.5, 2.0]:
        tr, _ = run_strategy_b(b60, band=band, stop_mult=s)
        R = tr["pips_net"] / tr["risk_pips"]
        grid.append({"band": band, "stop": s, "trades": len(tr),
                     "win_rate": (tr.pips_net > 0).mean(),
                     "avg_R": R.mean(),
                     "avg_R_gross": (tr["pips_gross"] / tr["risk_pips"]).mean()})
grid = pd.DataFrame(grid).round(3)
print(grid.pivot(index="band", columns="stop", values="avg_R"))
grid.sort_values("avg_R", ascending=False)

stop    1.0    1.5    2.0
band                     
1.5  -0.123 -0.076 -0.032
2.0  -0.187 -0.133 -0.131
2.5  -0.117 -0.024 -0.059


,band,stop,trades,win_rate,avg_R,avg_R_gross
7,2.5,1.5,59,0.492,-0.024,0.064
2,1.5,2.0,532,0.530,-0.032,0.032
8,2.5,2.0,59,0.525,-0.059,0.007
1,1.5,1.5,550,0.476,-0.076,0.009
6,2.5,1.0,59,0.373,-0.117,0.014
0,1.5,1.0,588,0.388,-0.123,0.005
5,2.0,2.0,185,0.465,-0.131,-0.063
4,2.0,1.5,189,0.429,-0.133,-0.043
3,2.0,1.0,193,0.347,-0.187,-0.051
